# Feature Engineering Challenge

## Challenge Objective

The objective of this challenge is to apply feature engineering and feature selection techniques to a real-world business problem.

We will work with a telecom customer dataset and develop a machine learning model to predict customer churn.

Customer churn means a customer stops using a company's services.

## Business Problem

A telecom company is experiencing customer loss.

The company wants to identify customers who are likely to leave so that it can take preventive actions.

Our task is to prepare the customer dataset and develop a machine learning model.

## Challenge Requirements

### Task 1: Understand the Dataset
- Inspect the dataset.
- Identify numerical and categorical columns.
- Check missing values.
- Check duplicate records.
- Understand the target variable.

### Task 2: Perform Feature Engineering

Create meaningful features using:
- Numerical transformations.
- Categorical transformations.
- Binning and discretization.
- Interaction features.
- Domain knowledge.

### Task 3: Justify Every Important Feature

Document:
- Feature name.
- Source columns.
- Feature logic.
- Reason for creation.
- Expected relationship with churn.
- Business meaning.
- Potential ML value.
- Leakage risk.
- Final decision.

### Task 4: Perform Feature Selection

Identify useful features and remove unnecessary information.

Methods may include:
- Correlation analysis.
- Statistical feature selection.
- Feature importance.

### Task 5: Prevent Feature Leakage

Ensure that:
- Target information is not included as an input.
- Future information is not used.
- Preprocessing is fitted only on training data.
- Test data remains unseen during model training.

### Task 6: Train and Evaluate the Model

Train a machine learning model and evaluate its performance using:

- Accuracy.
- Precision.
- Recall.
- F1-score.
- Confusion matrix.

### Task 7: Compare Results

Compare the model trained using original features with the model trained using engineered features.

## Expected Deliverables

1. Inspected dataset.
2. Engineered feature dataset.
3. Feature justification documentation.
4. Feature selection results.
5. Leakage prevention analysis.
6. Trained machine learning model.
7. Model performance comparison.
8. Final conclusion.

## Final Goal

Develop a reliable machine learning workflow that uses meaningful business features rather than simply increasing the number of columns.

# 2. Feature Engineering Challenge Demonstration

## Business Scenario: Telecom Customer Churn

A telecom company provides the following customer information:

| Feature | Description |
|---|---|
| CustomerID | Unique customer identifier |
| Age | Customer age |
| Tenure | Number of months as a customer |
| MonthlyCharges | Monthly customer charges |
| TotalCharges | Total customer charges |
| SupportCalls | Number of support calls |
| ContractType | Contract category |
| InternetService | Internet service category |
| PhoneService | Phone service status |
| OnlineSecurity | Online security subscription |
| Churn | Customer churn status |

## Our Engineering Challenge

We will create the following additional features:

| Feature | Source Columns | Business Purpose |
|---|---|---|
| Average_Monthly_Spending | TotalCharges, Tenure | Understand average spending |
| Customer_Loyalty_Group | Tenure | Identify customer loyalty |
| High_Value_Customer | MonthlyCharges | Identify high-revenue customers |
| Support_Issue_Level | SupportCalls | Identify possible customer service problems |
| Service_Count | InternetService, PhoneService, OnlineSecurity | Understand service adoption |
| Charges_Per_Service | MonthlyCharges, Service_Count | Understand average charges per service |
| Tenure_Charge_Interaction | Tenure, MonthlyCharges | Identify combined customer spending and relationship patterns |

## Model Comparison

We will build two models:

**Model 1 – Original Features**
- Uses the original customer information.
- Does not use the newly engineered features.

**Model 2 – Engineered Features**
- Uses the original information and additional engineered features.
- Uses preprocessing and feature selection.

We will compare both models using the same training and testing records.

## Important Observation

Feature engineering does not guarantee that model accuracy will always increase.

Some features may provide useful information, while others may be redundant or unnecessary.

The final decision should depend on model evaluation, business meaning, and data quality.

In [1]:
# ================================================================
# NOTEBOOK 16: FEATURE ENGINEERING CHALLENGE
# Business Problem: Telecom Customer Churn Prediction
# ================================================================


# STEP 1: IMPORT LIBRARIES

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")


# STEP 2: CREATE TELECOM CUSTOMER DATASET

np.random.seed(42)

n = 800

data = pd.DataFrame({
    "CustomerID": range(1, n + 1),
    "Age": np.random.randint(18, 75, n),
    "Tenure": np.random.randint(1, 73, n),
    "MonthlyCharges": np.random.uniform(20, 120, n).round(2),
    "SupportCalls": np.random.randint(0, 10, n),
    "ContractType": np.random.choice(
        ["Monthly", "Yearly", "Two Year"],
        n,
        p=[0.5, 0.3, 0.2]
    ),
    "InternetService": np.random.choice(
        ["DSL", "Fiber", "No"],
        n
    ),
    "PhoneService": np.random.choice(
        ["Yes", "No"],
        n
    ),
    "OnlineSecurity": np.random.choice(
        ["Yes", "No"],
        n
    )
})

# Generate total charges

data["TotalCharges"] = (
    data["Tenure"] * data["MonthlyCharges"]
    + np.random.normal(0, 40, n)
).clip(lower=0).round(2)


# STEP 3: GENERATE CHURN TARGET

churn_probability = (
    0.10
    + (data["Tenure"] < 12) * 0.20
    + (data["MonthlyCharges"] > 90) * 0.15
    + (data["SupportCalls"] > 5) * 0.20
    + (data["ContractType"] == "Monthly") * 0.15
)

churn_probability = churn_probability.clip(0, 0.95)

data["Churn"] = np.random.binomial(
    1,
    churn_probability
)


# STEP 4: INTRODUCE MISSING VALUES

data.loc[5:14, "Age"] = np.nan
data.loc[20:29, "MonthlyCharges"] = np.nan
data.loc[30:39, "ContractType"] = np.nan
data.loc[40:49, "InternetService"] = np.nan

print("\nDataset Created Successfully!")


# STEP 5: DATA INSPECTION

print("\n========== DATA INSPECTION ==========")

print("\nFirst Five Records:")
print(data.head())

print("\nDataset Shape:")
print(data.shape)

print("\nData Types:")
print(data.dtypes)

print("\nMissing Values:")
print(data.isnull().sum())

print("\nDuplicate Records:")
print(data.duplicated().sum())

print("\nChurn Distribution:")
print(data["Churn"].value_counts())


# STEP 6: REMOVE DUPLICATE RECORDS

data = data.drop_duplicates().copy()

print("\nDataset Shape After Duplicate Removal:")
print(data.shape)


# STEP 7: CREATE ENGINEERED FEATURES

# Feature 1: Average Monthly Spending

data["Average_Monthly_Spending"] = np.where(
    data["Tenure"] > 0,
    data["TotalCharges"] / data["Tenure"],
    np.nan
)

# Feature 2: Customer Loyalty Group

data["Customer_Loyalty_Group"] = pd.cut(
    data["Tenure"],
    bins=[0, 12, 36, np.inf],
    labels=["New", "Regular", "Loyal"],
    include_lowest=True
)

# Feature 3: High Value Customer

data["High_Value_Customer"] = np.where(
    data["MonthlyCharges"] > 80,
    "Yes",
    "No"
)

# Feature 4: Support Issue Level

data["Support_Issue_Level"] = pd.cut(
    data["SupportCalls"],
    bins=[-1, 2, 5, np.inf],
    labels=["Low", "Medium", "High"]
)

# Feature 5: Service Count

data["Service_Count"] = (
    (data["InternetService"] != "No").astype(int)
    + (data["PhoneService"] == "Yes").astype(int)
    + (data["OnlineSecurity"] == "Yes").astype(int)
)

# Feature 6: Charges Per Service

data["Charges_Per_Service"] = (
    data["MonthlyCharges"] /
    data["Service_Count"].replace(0, np.nan)
)

# Feature 7: Tenure and Monthly Charges Interaction

data["Tenure_Charge_Interaction"] = (
    data["Tenure"] * data["MonthlyCharges"]
)

print("\nEngineered Features Created Successfully!")


# STEP 8: DISPLAY ENGINEERED DATASET

print("\nFirst Five Engineered Records:")
print(data.head())

print("\nEngineered Dataset Shape:")
print(data.shape)


# STEP 9: CREATE FEATURE JUSTIFICATION TABLE

feature_documentation = pd.DataFrame({
    "Feature_Name": [
        "Average_Monthly_Spending",
        "Customer_Loyalty_Group",
        "High_Value_Customer",
        "Support_Issue_Level",
        "Service_Count",
        "Charges_Per_Service",
        "Tenure_Charge_Interaction"
    ],
    "Source_Columns": [
        "TotalCharges, Tenure",
        "Tenure",
        "MonthlyCharges",
        "SupportCalls",
        "InternetService, PhoneService, OnlineSecurity",
        "MonthlyCharges, Service_Count",
        "Tenure, MonthlyCharges"
    ],
    "Reason": [
        "Understand average spending",
        "Identify customer loyalty",
        "Identify high-revenue customers",
        "Identify possible service problems",
        "Understand service adoption",
        "Calculate average charges per service",
        "Capture combined tenure and spending patterns"
    ],
    "Expected_Churn_Relationship": [
        "May vary",
        "New customers may churn more",
        "Higher charges may increase churn",
        "More support issues may increase churn",
        "May vary",
        "May vary",
        "May capture combined effects"
    ],
    "Leakage_Risk": [
        "Low if available at prediction time",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low if based on current information"
    ],
    "Final_Decision": [
        "Retain",
        "Retain",
        "Retain",
        "Retain",
        "Retain",
        "Validate",
        "Validate"
    ]
})

print("\n========== FEATURE DOCUMENTATION ==========")
print(feature_documentation.to_string(index=False))


# STEP 10: ANALYZE ENGINEERED FEATURES AGAINST CHURN

print("\nChurn Rate by Customer Loyalty Group:")

print(
    data.groupby(
        "Customer_Loyalty_Group",
        observed=False
    )["Churn"].mean().round(3)
)

print("\nChurn Rate by Support Issue Level:")

print(
    data.groupby(
        "Support_Issue_Level",
        observed=False
    )["Churn"].mean().round(3)
)


# STEP 11: SEPARATE ORIGINAL AND ENGINEERED FEATURES

original_features = [
    "Age",
    "Tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SupportCalls",
    "ContractType",
    "InternetService",
    "PhoneService",
    "OnlineSecurity"
]

engineered_features = original_features + [
    "Average_Monthly_Spending",
    "Customer_Loyalty_Group",
    "High_Value_Customer",
    "Support_Issue_Level",
    "Service_Count",
    "Charges_Per_Service",
    "Tenure_Charge_Interaction"
]

X_original = data[original_features].copy()

X_engineered = data[engineered_features].copy()

y = data["Churn"].copy()

print("\nOriginal Feature Shape:", X_original.shape)
print("Engineered Feature Shape:", X_engineered.shape)


# STEP 12: SPLIT DATA INTO TRAINING AND TESTING SETS

# Use the same split for both model comparisons.

X_original_train, X_original_test, y_train, y_test = (
    train_test_split(
        X_original,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

X_engineered_train = X_engineered.loc[
    X_original_train.index
]

X_engineered_test = X_engineered.loc[
    X_original_test.index
]

print("\nTraining Records:", len(X_original_train))
print("Testing Records:", len(X_original_test))


# STEP 13: IDENTIFY COLUMN TYPES

def identify_columns(X):

    numerical_columns = X.select_dtypes(
        include=["int64", "float64"]
    ).columns.tolist()

    categorical_columns = X.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    return numerical_columns, categorical_columns


original_num, original_cat = identify_columns(
    X_original_train
)

engineered_num, engineered_cat = identify_columns(
    X_engineered_train
)


# STEP 14: CREATE REUSABLE PREPROCESSING FUNCTION

def create_preprocessor(numerical_columns, categorical_columns):

    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("numerical", numerical_pipeline, numerical_columns),
        ("categorical", categorical_pipeline, categorical_columns)
    ])

    return preprocessor


# STEP 15: CREATE ORIGINAL FEATURE MODEL

original_preprocessor = create_preprocessor(
    original_num,
    original_cat
)

original_model = Pipeline([
    ("preprocessing", original_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

original_model.fit(
    X_original_train,
    y_train
)

original_predictions = original_model.predict(
    X_original_test
)

print("\nOriginal Feature Model Trained Successfully!")


# STEP 16: CREATE ENGINEERED FEATURE MODEL

engineered_preprocessor = create_preprocessor(
    engineered_num,
    engineered_cat
)

engineered_model = Pipeline([
    ("preprocessing", engineered_preprocessor),
    ("feature_selection", SelectKBest(
        score_func=f_classif,
        k=12
    )),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

engineered_model.fit(
    X_engineered_train,
    y_train
)

engineered_predictions = engineered_model.predict(
    X_engineered_test
)

print("Engineered Feature Model Trained Successfully!")


# STEP 17: EVALUATE BOTH MODELS

def evaluate_model(model_name, y_true, predictions):

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1_Score": f1_score(
            y_true, predictions, zero_division=0
        )
    }


original_results = evaluate_model(
    "Original Features",
    y_test,
    original_predictions
)

engineered_results = evaluate_model(
    "Engineered Features",
    y_test,
    engineered_predictions
)

comparison = pd.DataFrame([
    original_results,
    engineered_results
])

print("\n========== MODEL PERFORMANCE COMPARISON ==========")

print(comparison.round(4).to_string(index=False))


# STEP 18: DISPLAY CONFUSION MATRICES

print("\nOriginal Feature Model Confusion Matrix:")

print(confusion_matrix(
    y_test,
    original_predictions
))

print("\nEngineered Feature Model Confusion Matrix:")

print(confusion_matrix(
    y_test,
    engineered_predictions
))


# STEP 19: DISPLAY CLASSIFICATION REPORTS

print("\nOriginal Feature Model Report:")

print(classification_report(
    y_test,
    original_predictions,
    zero_division=0
))

print("\nEngineered Feature Model Report:")

print(classification_report(
    y_test,
    engineered_predictions,
    zero_division=0
))


# STEP 20: DISPLAY SELECTED FEATURES

fitted_preprocessor = engineered_model.named_steps[
    "preprocessing"
]

transformed_feature_names = (
    fitted_preprocessor.get_feature_names_out()
)

selected_indices = engineered_model.named_steps[
    "feature_selection"
].get_support(indices=True)

selected_features = transformed_feature_names[
    selected_indices
]

print("\n========== SELECTED FEATURES ==========")

for feature in selected_features:
    print("-", feature)


# STEP 21: FEATURE LEAKAGE CHECK

print("\n========== FEATURE LEAKAGE CHECK ==========")

print("Target variable included in input features:",
      "Churn" in engineered_features)

print("CustomerID included in input features:",
      "CustomerID" in engineered_features)

print("Future outcome information included:",
      "CancellationConfirmed" in engineered_features)

print("\nTarget and identifier are excluded from model inputs.")


# STEP 22: FINAL CHALLENGE SUMMARY

print("\n========== FINAL CHALLENGE SUMMARY ==========")

print("Original Dataset Records:", len(data))

print("Original Features:", len(original_features))

print("Engineered Features:", len(engineered_features))

print("New Features Created:",
      len(engineered_features) - len(original_features))

print("Training Records:", len(X_engineered_train))

print("Testing Records:", len(X_engineered_test))

print("Selected Features:", len(selected_features))

print("\nOriginal Model Accuracy:",
      round(original_results["Accuracy"], 4))

print("Engineered Model Accuracy:",
      round(engineered_results["Accuracy"], 4))

if engineered_results["F1_Score"] > original_results["F1_Score"]:
    print("\nEngineered model achieved a higher F1-score.")
elif engineered_results["F1_Score"] < original_results["F1_Score"]:
    print("\nOriginal model achieved a higher F1-score.")
else:
    print("\nBoth models achieved the same F1-score.")

print("\nFeature Engineering Challenge Completed Successfully!")

Libraries imported successfully!

Dataset Created Successfully!

========== DATA INSPECTION ==========

First Five Records:
   CustomerID   Age  Tenure  MonthlyCharges  SupportCalls ContractType  \
0           1  56.0      45           20.64             1       Yearly   
1           2  69.0      32           48.70             2      Monthly   
2           3  46.0      45           81.69             0      Monthly   
3           4  32.0      61          118.12             5      Monthly   
4           5  60.0      47           83.18             5      Monthly   

  InternetService PhoneService OnlineSecurity  TotalCharges  Churn  
0           Fiber          Yes             No        978.91      0  
1              No          Yes             No       1540.47      0  
2              No          Yes            Yes       3648.36      0  
3             DSL          Yes             No       7126.56      0  
4              No           No            Yes       3913.60      0  

Dataset Shape:
(

C:\Users\Akalya Senthil\AppData\Local\Temp\ipykernel_20712\3750534881.py:349: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(
C:\Users\Akalya Senthil\AppData\Local\Temp\ipykernel_20712\3750534881.py:349: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_


Original Feature Model Trained Successfully!
Engineered Feature Model Trained Successfully!

========== MODEL PERFORMANCE COMPARISON ==========
              Model  Accuracy  Precision  Recall  F1_Score
  Original Features    0.6625     0.4800  0.2264    0.3077
Engineered Features    0.6250     0.3939  0.2453    0.3023

Original Feature Model Confusion Matrix:
[[94 13]
 [41 12]]

Engineered Feature Model Confusion Matrix:
[[87 20]
 [40 13]]

Original Feature Model Report:
              precision    recall  f1-score   support

           0       0.70      0.88      0.78       107
           1       0.48      0.23      0.31        53

    accuracy                           0.66       160
   macro avg       0.59      0.55      0.54       160
weighted avg       0.62      0.66      0.62       160


Engineered Feature Model Report:
              precision    recall  f1-score   support

           0       0.69      0.81      0.74       107
           1       0.39      0.25      0.30        5

# Feature Justification

## 1. Feature Engineering Documentation

| Feature Name | Source Columns | Feature Logic / Formula | Reason for Creation | Expected Relationship with Churn | Business Meaning | ML Value | Leakage Risk | Final Decision |
|---|---|---|---|---|---|---|---|---|
| Average_Monthly_Spending | TotalCharges, Tenure | TotalCharges / Tenure | Calculate average spending | May vary with spending behaviour | Average monthly customer spending | Medium | Low if available at prediction time | Retain |
| Customer_Loyalty_Group | Tenure | Group tenure into New, Regular, Loyal | Identify customer relationship duration | New customers may have higher churn | Customer loyalty | High | Low | Retain |
| High_Value_Customer | MonthlyCharges | MonthlyCharges > 80 | Identify higher revenue customers | Higher charges may increase churn | High-revenue customer segment | Medium | Low | Retain |
| Support_Issue_Level | SupportCalls | Group support calls into Low, Medium, High | Identify service problems | More support issues may increase churn | Customer service experience | High | Low | Retain |
| Service_Count | InternetService, PhoneService, OnlineSecurity | Count active services | Understand service adoption | Relationship may vary | Number of subscribed services | Medium | Low | Retain |
| Charges_Per_Service | MonthlyCharges, Service_Count | MonthlyCharges / Service_Count | Understand average charges per service | Relationship may vary | Average service spending | Medium | Low | Validate |
| Tenure_Charge_Interaction | Tenure, MonthlyCharges | Tenure × MonthlyCharges | Capture combined feature relationship | Relationship may vary | Combined tenure and spending pattern | Medium | Low | Validate |

## 2. Feature Selection Justification

The challenge uses `SelectKBest` with the ANOVA F-test.

This method ranks features based on their statistical relationship with the target variable.

The 12 highest-scoring transformed features are selected for model training.

However, statistical scores alone do not prove that a feature is useful. Model performance and business knowledge must also be considered.

## 3. Model Comparison

Two models were developed:

### Model 1: Original Features

Uses the original customer information.

### Model 2: Engineered Features

Uses the original information and seven additional engineered features, followed by preprocessing and feature selection.

The models are compared using the same training and testing records.

Important evaluation metrics:

- Accuracy: Overall correct predictions.
- Precision: How many predicted churn customers actually churned.
- Recall: How many actual churn customers were identified.
- F1-score: Balance between precision and recall.

For customer churn prediction, recall is especially important because failing to identify customers who may leave can result in lost business.

## 4. Leakage Prevention Justification

The workflow follows these practices:

1. CustomerID is excluded from model inputs.
2. Churn is used only as the target.
3. No post-churn information is used.
4. Training and testing data are separated before preprocessing.
5. Imputation and scaling are fitted using training data.
6. Feature selection is performed inside the training pipeline.
7. Model evaluation uses unseen test data.

## 5. Final Business Interpretation

The engineered features provide additional information about:

- Customer loyalty.
- Customer spending.
- Service usage.
- Customer support problems.
- Customer service adoption.

These features can help the telecom company understand customer behaviour and identify customers who may need retention support.

## 6. Final Conclusion

Feature engineering is an important part of machine learning development.

The quality of features can influence how effectively a model identifies patterns.

However, creating more features does not automatically improve performance.

Each feature should be evaluated using:

- Business understanding.
- Statistical analysis.
- Model performance.
- Data quality.
- Leakage prevention.

**Final Rule:** Create meaningful features, justify their purpose, validate their usefulness, and retain only features that are appropriate for the prediction problem.